In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import pandas as pd
import numpy as np
import keras
from keras.utils import FeatureSpace
import gc

I0000 00:00:1794386241.396634    5515 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794386241.767430    5515 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794386244.149122    5515 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
gc.collect()
df1 = pd.read_csv("./df_knni_rounded.csv", sep='\t').copy().reset_index(drop = True)
df1

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.230,26.34,72.93,3.60246,...,401.0,0.0,12.0,24.0,1.0,2.0,1.0,2.0,1.0,2.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.120,44.83,86.71,3.36580,...,1094.0,0.0,0.0,49.0,8.0,3.0,4.0,0.0,1.0,3.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,40.086,33.65,72.17,3.41520,...,645.0,0.0,17.0,48.0,3.0,2.0,4.0,2.0,0.0,6.0
3,2022.0,47.35,83700.0,6.44,353.0,85.11,54.320,37.73,82.88,3.42020,...,867.0,0.0,18.0,18.0,1.0,3.0,4.0,0.0,1.0,1.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.730,65.29,86.17,3.45560,...,1090.0,0.0,16.0,29.0,0.0,0.0,4.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,2016.0,45.99,182311.0,10.11,817.0,86.91,30.510,42.42,84.85,2.89010,...,2691.0,0.0,2.0,40.0,1.0,2.0,0.0,1.0,1.0,1.0
199996,2015.0,67.65,275317.0,11.06,840.0,65.08,77.610,30.76,65.08,3.57070,...,420.0,0.0,8.0,0.0,4.0,3.0,4.0,0.0,1.0,0.0
199997,2019.0,98.08,158747.0,4.89,336.0,86.08,44.590,24.64,84.90,2.98750,...,3148.0,0.0,2.0,40.0,0.0,4.0,0.0,1.0,0.0,2.0
199998,2016.0,51.49,95122.0,10.06,392.0,76.61,80.700,41.93,75.50,3.56980,...,797.0,0.0,16.0,30.0,7.0,0.0,4.0,1.0,1.0,1.0


In [3]:
np.random.seed(10)
df2 = df1.copy().reset_index(drop = True)
remove_n = 160000
drop_indices = np.random.choice(df2.index, remove_n, replace=False)
df2_subset = df2.drop(drop_indices)
df2_subset


,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
10,2020.0,66.82,88584.0,7.772,254.0,86.07,67.23,38.43,86.16,3.4856,...,1049.0,0.0,1.0,54.0,7.0,4.0,4.0,2.0,1.0,3.0
12,2023.0,48.14,17280.0,3.060,81.0,96.61,65.48,31.67,97.14,3.4579,...,1351.0,0.0,19.0,13.0,2.0,2.0,4.0,1.0,1.0,1.0
15,2020.0,59.96,4085.0,6.100,27.0,99.61,48.10,25.20,99.93,3.3970,...,1529.0,0.0,6.0,21.0,5.0,4.0,4.0,0.0,1.0,3.0
19,2015.0,48.25,185354.0,10.900,789.0,66.02,27.29,27.00,65.13,3.2647,...,452.0,0.0,8.0,27.0,1.0,0.0,4.0,0.0,0.0,1.0
21,2018.0,92.13,25401.0,7.860,73.0,99.80,11.43,33.19,100.00,3.2089,...,1445.0,0.0,1.0,53.0,1.0,1.0,4.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199972,2017.0,85.55,335843.0,9.030,793.0,59.21,52.62,40.55,58.73,3.4085,...,436.0,0.0,6.0,23.0,1.0,3.0,4.0,1.0,0.0,2.0
199973,2015.0,53.63,53728.0,10.750,115.0,89.94,50.99,32.50,92.99,3.4313,...,1269.0,0.0,12.0,24.0,3.0,3.0,4.0,1.0,1.0,3.0
199978,2020.0,58.79,55703.0,6.220,134.0,94.97,78.12,29.08,98.66,3.5252,...,1292.0,0.0,1.0,53.0,1.0,3.0,4.0,2.0,1.0,0.0
199980,2019.0,50.30,58184.0,7.140,263.0,92.27,100.00,20.21,93.76,3.6988,...,1108.0,0.0,6.0,11.0,1.0,2.0,4.0,2.0,1.0,3.0


In [4]:
class keras(object):

    def __init__(self, dataset):
        self.dataset = dataset

    def train_test_split(self):
        global val_dataframe_global
        global train_dataframe_global
        global df4_global
        
        gc.collect()
        df4 = self.dataset.copy().reset_index(drop = True)
        val_dataframe = df4.sample(frac=0.2, random_state=1337)
        train_dataframe = df4.drop(val_dataframe.index)
        val_dataframe_global = val_dataframe
        train_dataframe_global = train_dataframe
        df4_global = df4
        
        print(
            "Using %d samples for training and %d for validation"
            % (len(train_dataframe), len(val_dataframe)))

    def dataframe_to_dataset(self):
        global train_ds_global
        global train_labels_global
        global val_ds_global
        global val_labels_global

        gc.collect()
        train_labels = train_dataframe_global.pop("battery_failure")
        train_labels_global = train_labels
        train_ds = tf.data.Dataset.from_tensor_slices((dict(train_dataframe_global), train_labels))
        gc.collect()
        train_ds = train_ds.shuffle(buffer_size=len(train_dataframe_global))
        train_ds_global = train_ds
        gc.collect()
        val_labels = val_dataframe_global.pop("battery_failure")
        val_labels_global = val_labels
        val_ds = tf.data.Dataset.from_tensor_slices((dict(val_dataframe_global), val_labels))
        gc.collect()
        val_ds = val_ds.shuffle(buffer_size=len(val_dataframe_global))
        gc.collect()
        val_ds_global = val_ds

    def print_x_y(self):

        gc.collect()
        for x, y in train_ds_global.take(1):
            print("Input:", x)
            print("Target:", y)

    def dataset_batch(self):
        global train_ds_batch_global
        global val_ds_batch_global

        gc.collect()
        train_ds_batch_global = train_ds_global.batch(32)
        val_ds_batch_global = val_ds_global.batch(32)

    def feature_space(self):

        global feature_space_global

        gc.collect()
        feature_space = FeatureSpace(
            features={
                # Categorical features encoded as integers
                "manufacturing_year": "integer_categorical",
                #"battery_failure": "integer_categorical",
                "Vehicle_Brand_Num_Value": "integer_categorical",
                "Vehicle_Model_Num_Value": "integer_categorical",
                "Battery_Manufacturer_Num_Value": "integer_categorical",
                "Terrain_Type_Num_Value": "integer_categorical",
                "Battery_Chemistry_Num_Value": "integer_categorical",
                "Drive_Type_Num_Value": "integer_categorical",
                "Fleet_Or_Private_Num_Value": "integer_categorical",
                "Vehicle_Type_Num_Value": "integer_categorical",
                # Categorical feature encoded as string
                
                # Numerical features to discretize
                "vehicle_age_years": "float_discretized",
                # Numerical features to normalize
                "battery_capacity_kwh": "float_normalized",
                "odometer_km": "float_normalized",
                "cycle_count": "float_normalized",
                "battery_health_percent": "float_normalized",
                "state_of_charge": "float_normalized",
                "depth_of_discharge": "float_normalized",
                "state_of_health": "float_normalized",
                "cell_voltage_avg": "float_normalized",
                "cell_voltage_std": "float_normalized",
                "pack_voltage": "float_normalized",
                "cell_temperature_avg": "float_normalized",
                "cell_temperature_max": "float_normalized",
                "internal_resistance": "float_normalized",
                "charge_efficiency": "float_normalized",
                "discharge_efficiency": "float_normalized",
                "remaining_capacity": "float_normalized",
                "capacity_loss_percent": "float_normalized",
                "charging_cycles_last_month": "float_normalized",
                "fast_charge_ratio": "float_normalized",
                "slow_charge_ratio": "float_normalized",
                "average_charge_power_kw": "float_normalized",
                "average_charging_time": "float_normalized",
                "overnight_charging_ratio": "float_normalized",
                "home_charging_ratio": "float_normalized",
                "charging_interruptions": "float_normalized",
                "overcharge_events": "float_normalized",
                "average_speed": "float_normalized",
                "average_trip_distance": "float_normalized",
                "aggressive_acceleration_score": "float_normalized",
                "hard_braking_score": "float_normalized",
                "regenerative_braking_usage": "float_normalized",
                "highway_driving_ratio": "float_normalized",
                "city_driving_ratio": "float_normalized",
                "daily_distance": "float_normalized",
                "average_ambient_temperature": "float_normalized",
                "maximum_temperature": "float_normalized",
                "minimum_temperature": "float_normalized",
                "humidity": "float_normalized",
                "altitude": "float_normalized",
                "dust_exposure": "float_normalized",
                "last_service_days": "float_normalized",
                "cooling_system_health": "float_normalized",
                "firmware_updates": "float_normalized",
                "previous_faults": "float_normalized",
                "maintenance_score": "float_normalized",
                "thermal_runaway_risk": "float_normalized",
                "voltage_imbalance": "float_normalized",
                "temperature_variance": "float_normalized",
                "sensor_fault_count": "float_normalized",
                "BMS_warning_count": "float_normalized",
                "abnormal_voltage_events": "float_normalized",
                "battery_stress_index": "float_normalized",
                "aging_score": "float_normalized",
                "thermal_health_score": "float_normalized",
                "charging_quality_score": "float_normalized",
                "driving_stress_score": "float_normalized",
                "predicted_remaining_life_cycles": "float_normalized"
            },
            # We create additional features by hashing
            # value co-occurrences for the
            # following groups of categorical features.
            crosses=[("manufacturing_year", "Battery_Manufacturer_Num_Value"), ("vehicle_age_years", "Vehicle_Model_Num_Value")],
            # The hashing space for these co-occurrences
            # wil be 32-dimensional.
            crossing_dim=32,
            # Our utility will one-hot encode all categorical
            # features and concat all features into a single
            # vector (one vector per sample).
            output_mode="concat",)

        feature_space_global = feature_space


    def adapt_featuresapce(self):
        global train_ds_with_no_labels
        global preprocessed_x_global
        
        gc.collect()
        train_ds_with_no_labels = train_ds_batch_global.map(lambda x, _: x)
        feature_space_global.adapt(train_ds_with_no_labels)

        train_ds_with_no_labels_global = train_ds_with_no_labels

        for x, _ in train_ds_batch_global.take(1):
            preprocessed_x = feature_space_global(x)
            preprocessed_x_global = preprocessed_x
            print("preprocessed_x.shape:", preprocessed_x.shape)
            print("preprocessed_x.dtype:", preprocessed_x.dtype)

    def training_validation_dataset_of_preprocessed_batches(self):

        global preprocessed_train_ds_global
        global preprocessed_val_ds_global

        gc.collect()
        preprocessed_train_ds = train_ds_batch_global.map(
        lambda x, y: (feature_space_global(x), y), num_parallel_calls=tf.data.AUTOTUNE)
        preprocessed_train_ds = preprocessed_train_ds.prefetch(tf.data.AUTOTUNE)

        preprocessed_val_ds = val_ds_batch_global.map(
            lambda x, y: (feature_space_global(x), y), num_parallel_calls=tf.data.AUTOTUNE)

        preprocessed_val_ds = preprocessed_val_ds.prefetch(tf.data.AUTOTUNE)

        preprocessed_train_ds_global = preprocessed_train_ds
        preprocessed_val_ds_global = preprocessed_val_ds

    def building_model(self):

        global training_model_global
        global inference_model_global

        gc.collect()
        dict_inputs = feature_space_global.get_inputs()
        encoded_features = feature_space_global.get_encoded_features()

        x = tf.keras.layers.Dense(32, activation="relu")(encoded_features)
        x = tf.keras.layers.Dropout(0.5)(x)
        predictions = tf.keras.layers.Dense(1, activation="sigmoid")(x)

        training_model = tf.keras.Model(inputs=encoded_features, outputs=predictions)
        training_model.compile(
            optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        inference_model = tf.keras.Model(inputs=dict_inputs, outputs=predictions)

        training_model_global = training_model
        inference_model_global = inference_model

    def training_model(self):

        gc.collect()
        training_model_global.fit(
            preprocessed_train_ds_global,
            epochs=20,
            validation_data=preprocessed_val_ds_global,
            verbose=2,)


    def main(self):

        gc.collect()
        self.train_test_split()
        gc.collect()
        self.dataframe_to_dataset()
        gc.collect()
        self.print_x_y()
        gc.collect()
        self.dataset_batch()
        gc.collect()
        self.feature_space()
        gc.collect()
        self.adapt_featuresapce()
        gc.collect()
        self.training_validation_dataset_of_preprocessed_batches()
        gc.collect()
        self.building_model()
        gc.collect()
        self.training_model()
        gc.collect()


        


        

        

In [5]:
keras_01 = keras(df2_subset)
keras_01.main()

Using 32000 samples for training and 8000 for validation


E0000 00:00:1794386249.855267    5515 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



Input: {'manufacturing_year': <tf.Tensor: shape=(), dtype=float64, numpy=2022.0>, 'battery_capacity_kwh': <tf.Tensor: shape=(), dtype=float64, numpy=71.53>, 'odometer_km': <tf.Tensor: shape=(), dtype=float64, numpy=117724.0>, 'vehicle_age_years': <tf.Tensor: shape=(), dtype=float64, numpy=3.97>, 'cycle_count': <tf.Tensor: shape=(), dtype=float64, numpy=376.0>, 'battery_health_percent': <tf.Tensor: shape=(), dtype=float64, numpy=80.89>, 'state_of_charge': <tf.Tensor: shape=(), dtype=float64, numpy=20.4>, 'depth_of_discharge': <tf.Tensor: shape=(), dtype=float64, numpy=46.194>, 'state_of_health': <tf.Tensor: shape=(), dtype=float64, numpy=82.28>, 'cell_voltage_avg': <tf.Tensor: shape=(), dtype=float64, numpy=3.2334>, 'cell_voltage_std': <tf.Tensor: shape=(), dtype=float64, numpy=0.01848>, 'pack_voltage': <tf.Tensor: shape=(), dtype=float64, numpy=808.36>, 'cell_temperature_avg': <tf.Tensor: shape=(), dtype=float64, numpy=2.85>, 'cell_temperature_max': <tf.Tensor: shape=(), dtype=float64,

/home/xy/Desktop/ml/tensorflow/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1000/1000 - 6s - 6ms/step - accuracy: 0.9477 - loss: 0.1295 - val_accuracy: 0.9709 - val_loss: 0.0727
Epoch 2/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9648 - loss: 0.0869 - val_accuracy: 0.9709 - val_loss: 0.0706
Epoch 3/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9667 - loss: 0.0808 - val_accuracy: 0.9714 - val_loss: 0.0700
Epoch 4/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9682 - loss: 0.0796 - val_accuracy: 0.9712 - val_loss: 0.0706
Epoch 5/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9692 - loss: 0.0772 - val_accuracy: 0.9709 - val_loss: 0.0690
Epoch 6/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9702 - loss: 0.0751 - val_accuracy: 0.9712 - val_loss: 0.0683
Epoch 7/20
1000/1000 - 4s - 4ms/step - accuracy: 0.9708 - loss: 0.0738 - val_accuracy: 0.9711 - val_loss: 0.0699
Epoch 8/20
1000/1000 - 5s - 5ms/step - accuracy: 0.9703 - loss: 0.0729 - val_accuracy: 0.9691 - val_loss: 0.0709
Epoch 9/20
1000/1000 - 5s - 5ms/step - accuracy: 0.9722 - loss: 0.0710 - val_accuracy: 0.9700 - val_loss: 0

In [8]:
sample = {
    'manufacturing_year': 2016.00000,
    'battery_capacity_kwh': 120.34000,
    'odometer_km':  5427100.00000,
    'vehicle_age_years':  10.13000,
    'cycle_count':  1.00000,
    'battery_health_percent':  10.67000,
    'state_of_charge': 21.10000,
    'depth_of_discharge': 89.00000,
    'state_of_health': 10.01000,
    'cell_voltage_avg': 2.20910,
    'cell_voltage_std': 0.01757,
    'pack_voltage': 552.26000,
    'cell_temperature_avg': 44.39000,
    'cell_temperature_max': 50.77000,
    'internal_resistance': 0.31940,
    'charge_efficiency': 95.90000,
    'discharge_efficiency': 95.40000,
    'remaining_capacity': 62.87000,
    'capacity_loss_percent': 9.33000,
    'charging_cycles_last_month': 1.00000,
    'fast_charge_ratio': 0.44600,
    'slow_charge_ratio': 0.56700,
    'average_charge_power_kw': 41.30000,
    'average_charging_time': 100.40000,
    'overnight_charging_ratio': 0.43100,
    'home_charging_ratio': 0.75200,
    'charging_interruptions': 0.00000,
    'overcharge_events': 1.00000,
    'average_speed': 50.90000,
    'average_trip_distance': 4.10000,
    'aggressive_acceleration_score': 46.80000,
    'hard_braking_score': 33.20000,
    'regenerative_braking_usage': 44.10000,
    'highway_driving_ratio': 0.32300,
    'city_driving_ratio': 0.65300,
    'daily_distance': 13.20000,
    'average_ambient_temperature': 37.50000,
    'maximum_temperature': 49.90000,
    'minimum_temperature': 31.60000,
    'humidity': 58.50000,
    'altitude': 594.70000,
    'dust_exposure': 41.10000,
    'last_service_days': 173.00000,
    'cooling_system_health': 45.30000,
    'firmware_updates': 8.00000,
    'previous_faults': 4.00000,
    'maintenance_score': 81.30000,
    'thermal_runaway_risk': 23.30000,
    'voltage_imbalance': 22.08000,
    'temperature_variance': 12.39000,
    'sensor_fault_count': 2.00000,
    'BMS_warning_count': 0.00000,
    'abnormal_voltage_events': 1.00000,
    'battery_stress_index': 75.80000,
    'aging_score': 23.80000,
    'thermal_health_score': 10.60000,
    'charging_quality_score': 10.80000,
    'driving_stress_score': 90.60000,
    'predicted_remaining_life_cycles': 30.00000,
    'Vehicle_Brand_Num_Value': 10.00000,
    'Vehicle_Model_Num_Value': 44.00000,
    'Battery_Manufacturer_Num_Value': 4.00000,
    "Terrain_Type_Num_Value": 2,
    "Battery_Chemistry_Num_Value": 1,
    "Drive_Type_Num_Value": 1,
    "Fleet_Or_Private_Num_Value": 0,
    "Vehicle_Type_Num_Value": 1
}

input_dict = {name: tf.convert_to_tensor([value]) for name, value in sample.items()}
predictions = inference_model_global.predict(input_dict)

print(
    f"This particular car has a {100 * predictions[0][0]:.2f}% probability "
    "of having a battery failure, as evaluated by our model."
)


/home/xy/Desktop/ml/tensorflow/lib/python3.14/site-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'BMS_warning_count': 'BMS_warning_count', 'Battery_Chemistry_Num_Value': 'Battery_Chemistry_Num_Value', 'Battery_Manufacturer_Num_Value': 'Battery_Manufacturer_Num_Value', 'Drive_Type_Num_Value': 'Drive_Type_Num_Value', 'Fleet_Or_Private_Num_Value': 'Fleet_Or_Private_Num_Value', 'Terrain_Type_Num_Value': 'Terrain_Type_Num_Value', 'Vehicle_Brand_Num_Value': 'Vehicle_Brand_Num_Value', 'Vehicle_Model_Num_Value': 'Vehicle_Model_Num_Value', 'Vehicle_Type_Num_Value': 'Vehicle_Type_Num_Value', 'abnormal_voltage_events': 'abnormal_voltage_events', 'aggressive_acceleration_score': 'aggressive_acceleration_score', 'aging_score': 'aging_score', 'altitude': 'altitude', 'average_ambient_temperature': 'average_ambient_temperature', 'average_charge_power_kw': 'average_charge_power_kw', 'average_charging_time': 'average_

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 869ms/step
This particular car has a 100.00% probability of having a battery failure, as evaluated by our model.


In [9]:
gc.collect()

4446